<a href="https://colab.research.google.com/github/elyadaeben/IRDR0001-Coursework/blob/main/TS_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task
Download the dataset from "https://zenodo.org/records/3660927", extract the data for Jakarta, and then summarize the extracted Jakarta data.

## Download Data

### Subtask:
Download the dataset from the provided URL (https://zenodo.org/records/3660927) using `wget`.


**Reasoning**:
The subtask requires downloading a file using `wget` and then extracting its contents using `unzip`. This can be achieved with shell commands within a code block.



# Task
Download the NetCDF dataset from "https://zenodo.org/records/3660927/files/CODEC_amax_ERA5_1979_2017_coor_mask_GUM_RPS.nc?download=1", extract the extreme sea levels, tides, and storm surge data specifically for Jakarta, and then summarize the extracted data.

## Download Data

### Subtask:
Download the NetCDF dataset from the provided URL (https://zenodo.org/records/3660927/files/CODEC_amax_ERA5_1979_2017_coor_mask_GUM_RPS.nc?download=1) using `wget`.


**Reasoning**:
The subtask requires downloading a NetCDF file using `wget`. This can be achieved with a shell command within a code block.



In [ ]:
!wget -O CODEC_amax_ERA5_1979_2017_coor_mask_GUM_RPS.nc "https://zenodo.org/records/3660927/files/CODEC_amax_ERA5_1979_2017_coor_mask_GUM_RPS.nc?download=1"


--2026-01-05 19:40:04--  https://zenodo.org/records/3660927/files/CODEC_amax_ERA5_1979_2017_coor_mask_GUM_RPS.nc?download=1
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 137.138.52.235, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3291991 (3.1M) [application/octet-stream]
Saving to: ‘CODEC_amax_ERA5_1979_2017_coor_mask_GUM_RPS.nc’

CODEC_amax_ERA5_197 100%[===================>]   3.14M  1.34MB/s    in 2.3s    

2026-01-05 19:40:07 (1.34 MB/s) - ‘CODEC_amax_ERA5_1979_2017_coor_mask_GUM_RPS.nc’ saved [3291991/3291991]



## Load NetCDF Data

### Subtask:
Load the downloaded NetCDF file into a suitable data structure, likely using a library like `xarray` for handling geospatial data.


**Reasoning**:
To load the NetCDF data, I need to import the `xarray` library, open the specified file, and then display its contents to confirm it's loaded correctly.



In [ ]:
import xarray as xr

ds = xr.open_dataset('CODEC_amax_ERA5_1979_2017_coor_mask_GUM_RPS.nc')
print(ds)

<xarray.Dataset> Size: 3MB
Dimensions:               (stations: 14110, return_periods: 9,
                           gumbel_parameters: 2)
Coordinates:
  * return_periods        (return_periods) float64 72B 2.0 5.0 ... 500.0 1e+03
  * gumbel_parameters     (gumbel_parameters) |S64 128B b'location           ...
    station_y_coordinate  (stations) float64 113kB ...
    station_x_coordinate  (stations) float64 113kB ...
    station_name          (stations) |S64 903kB ...
Dimensions without coordinates: stations
Data variables:
    station_id            (stations) |S64 903kB ...
    RPS                   (stations, return_periods) float64 1MB ...
    GUM                   (stations, gumbel_parameters) float64 226kB ...
Attributes:
    CDI:            Climate Data Interface version 1.9.5 (http://mpimet.mpg.d...
    Conventions:    CF-1.5 Deltares-0.1
    history:        Thu Sep 19 11:22:14 2019: cdo -L -yearmax -selyear,1979 /...
    source:         Deltares, D-Flow FM Version 1.1.270.5410

## Extract Jakarta Data

### Subtask:
Identify the geographical coordinates for Jakarta and filter the loaded dataset to extract the extreme sea levels, tides, and storm surge data specifically for that area.


**Reasoning**:
To identify the Jakarta station, I need to examine the 'station_name' variable within the `ds` dataset. I will first convert the byte strings to regular strings for easier searching and then display some unique names or search for 'Jakarta' to understand the available station names.



In [ ]:
station_names_decoded = ds['station_name'].values.astype(str)
jkt_stations_mask = [b'Jakarta' in name for name in ds['station_name'].values]

if any(jkt_stations_mask):
    print("Found Jakarta stations:")
    print(ds['station_name'].values[jkt_stations_mask].astype(str))
    jakarta_ds = ds.isel(stations=jkt_stations_mask)
    print("\nFiltered dataset for Jakarta:")
    print(jakarta_ds)
else:
    print("Jakarta station not found. Displaying some station names for inspection:")
    print(station_names_decoded[:10])

Jakarta station not found. Displaying some station names for inspection:
['id_coast_glob_eur_00001' 'id_coast_glob_eur_00002'
 'id_coast_glob_eur_00003' 'id_coast_glob_eur_00004'
 'id_coast_glob_eur_00005' 'id_coast_glob_eur_00006'
 'id_coast_glob_eur_00007' 'id_coast_glob_eur_00008'
 'id_coast_glob_eur_00009' 'id_coast_glob_eur_00010']


**Reasoning**:
Since direct searching for 'Jakarta' in station names did not yield results, I will identify the geographical coordinates for Jakarta and then filter the dataset based on these coordinates. I will start by using widely known approximate coordinates for Jakarta and then refine the search by looking for stations within a certain radius of these coordinates.



In [ ]:
jakarta_lat = -6.2
jakarta_lon = 106.8

# Define a search radius (e.g., 0.5 degrees latitude/longitude)
radius_lat = 0.5
radius_lon = 0.5

# Create a mask for stations within the Jakarta region
lat_mask = (ds['station_y_coordinate'] >= (jakarta_lat - radius_lat)) & \
           (ds['station_y_coordinate'] <= (jakarta_lat + radius_lat))
lon_mask = (ds['station_x_coordinate'] >= (jakarta_lon - radius_lon)) & \
           (ds['station_x_coordinate'] <= (jakarta_lon + radius_lon))

jkt_stations_mask_coords = lat_mask & lon_mask

if jkt_stations_mask_coords.any():
    jakarta_ds = ds.isel(stations=jkt_stations_mask_coords)
    print("Filtered dataset for Jakarta based on coordinates:")
    print(jakarta_ds)
    print("\nStation names found near Jakarta coordinates:")
    print(jakarta_ds['station_name'].values.astype(str))
else:
    print("No stations found near Jakarta's coordinates. Please adjust radius or coordinates if needed.")

Filtered dataset for Jakarta based on coordinates:
<xarray.Dataset> Size: 1kB
Dimensions:               (stations: 5, return_periods: 9, gumbel_parameters: 2)
Coordinates:
  * return_periods        (return_periods) float64 72B 2.0 5.0 ... 500.0 1e+03
  * gumbel_parameters     (gumbel_parameters) |S64 128B b'location           ...
    station_y_coordinate  (stations) float64 40B -6.021 -6.021 ... -5.962 -5.962
    station_x_coordinate  (stations) float64 40B 106.5 106.7 106.9 107.0 107.2
    station_name          (stations) |S64 320B b'id_coast_glob_12946' ... b'i...
Dimensions without coordinates: stations
Data variables:
    station_id            (stations) |S64 320B ...
    RPS                   (stations, return_periods) float64 360B ...
    GUM                   (stations, gumbel_parameters) float64 80B ...
Attributes:
    CDI:            Climate Data Interface version 1.9.5 (http://mpimet.mpg.d...
    Conventions:    CF-1.5 Deltares-0.1
    history:        Thu Sep 19 11:22:14 2019

**Reasoning**:
The subtask is to extract and summarize Jakarta data. Having successfully extracted the data for Jakarta into `jakarta_ds`, the next step is to summarize this data to understand its key characteristics.



In [ ]:
print("Summary of Jakarta Data (RPS variable):")
print(jakarta_ds['RPS'].describe())
print("\nSummary of Jakarta Data (GUM variable):")
print(jakarta_ds['GUM'].describe())

Summary of Jakarta Data (RPS variable):


AttributeError: 'DataArray' object has no attribute 'describe'

In [ ]:
print(ds.attrs)

{'CDI': 'Climate Data Interface version 1.9.5 (http://mpimet.mpg.de/cdi)', 'Conventions': 'CF-1.5 Deltares-0.1', 'history': 'Thu Sep 19 11:22:14 2019: cdo -L -yearmax -selyear,1979 /p/11200665-c3s-codec/2_Hydro/1_RUNS/2_ECMWF_ERA5/tide_surge/model_input_dataMetOcean_1979_FM_corr_new/output/gtsm_fine_0000_his.nc /p/11200665-c3s-codec/2_Hydro/1_RUNS/2_ECMWF_ERA5/tide_surge/model_input_dataMetOcean_1979_FM_corr_new/output/gtsm_fine_0000_his_GLOB_YRMAX_v2.nc\nCreated on 2019-03-06T17:18:41+0100, D-Flow FM', 'source': 'Deltares, D-Flow FM Version 1.1.270.54102, Jan 16 2018, 15:14:11, model', 'institution': 'Deltares', 'references': 'http://www.deltares.nl', 'date_created': '2019-03-06T17:18:41+0100', 'date_modified': '2019-03-06T17:18:41+0100', 'frequency': 'year', 'CDO': 'Climate Data Operators version 1.9.5 (http://mpimet.mpg.de/cdo)'}


Please check the Zenodo record for more detailed documentation regarding the datum of this dataset:

[https://zenodo.org/records/3660927](https://zenodo.org/records/3660927)

**Reasoning**:
The previous code failed because `xarray.DataArray` objects do not have a `describe()` method like pandas DataFrames. To summarize the data, I will explicitly calculate common statistics such as mean, min, max, and standard deviation using available xarray methods.



In [ ]:
print("Summary of Jakarta Data (RPS variable):")
print(f"Mean: {jakarta_ds['RPS'].mean().values}")
print(f"Min: {jakarta_ds['RPS'].min().values}")
print(f"Max: {jakarta_ds['RPS'].max().values}")
print(f"Standard Deviation: {jakarta_ds['RPS'].std().values}")

print("\nSummary of Jakarta Data (GUM variable):")
print(f"Mean: {jakarta_ds['GUM'].mean().values}")
print(f"Min: {jakarta_ds['GUM'].min().values}")
print(f"Max: {jakarta_ds['GUM'].max().values}")
print(f"Standard Deviation: {jakarta_ds['GUM'].std().values}")

Summary of Jakarta Data (RPS variable):
Mean: 1.047684690694655
Min: 0.8239915992233
Max: 1.192540930381515
Standard Deviation: 0.08786931059709056

Summary of Jakarta Data (GUM variable):
Mean: 0.4706360629799253
Min: 0.032382157613751114
Max: 0.941568607733481
Standard Deviation: 0.43440318086758717


In [ ]:
rp_100_data = jakarta_ds.sel(return_periods=100, method='nearest')

print("Data for Return Period 100:")
print(rp_100_data)

print("\nSummary of RPS for Return Period 100:")
print(f"Mean: {rp_100_data['RPS'].mean().values}")
print(f"Min: {rp_100_data['RPS'].min().values}")
print(f"Max: {rp_100_data['RPS'].max().values}")
print(f"Standard Deviation: {rp_100_data['RPS'].std().values}")

print("\nSummary of GUM for Return Period 100:")
print(f"Mean: {rp_100_data['GUM'].mean().values}")
print(f"Min: {rp_100_data['GUM'].min().values}")
print(f"Max: {rp_100_data['GUM'].max().values}")
print(f"Standard Deviation: {rp_100_data['GUM'].std().values}")

Data for Return Period 100:
<xarray.Dataset> Size: 976B
Dimensions:               (stations: 5, gumbel_parameters: 2)
Coordinates:
  * gumbel_parameters     (gumbel_parameters) |S64 128B b'location           ...
    return_periods        float64 8B 100.0
    station_y_coordinate  (stations) float64 40B -6.021 -6.021 ... -5.962 -5.962
    station_x_coordinate  (stations) float64 40B 106.5 106.7 106.9 107.0 107.2
    station_name          (stations) |S64 320B b'id_coast_glob_12946' ... b'i...
Dimensions without coordinates: stations
Data variables:
    station_id            (stations) |S64 320B ...
    RPS                   (stations) float64 40B 1.091 1.103 1.105 1.082 1.003
    GUM                   (stations, gumbel_parameters) float64 80B 0.9416 .....
Attributes:
    CDI:            Climate Data Interface version 1.9.5 (http://mpimet.mpg.d...
    Conventions:    CF-1.5 Deltares-0.1
    history:        Thu Sep 19 11:22:14 2019: cdo -L -yearmax -selyear,1979 /...
    source:         De

In [ ]:
print(raster_ds)

NameError: name 'raster_ds' is not defined

In [ ]:
print("Measurements from Nearest Stations (RP 100):")
station_data_rp100 = rp_100_data[['station_name', 'station_x_coordinate', 'station_y_coordinate', 'RPS']]
print(station_data_rp100)

Measurements from Nearest Stations (RP 100):
<xarray.Dataset> Size: 448B
Dimensions:               (stations: 5)
Coordinates:
    station_name          (stations) |S64 320B b'id_coast_glob_12946' ... b'i...
    station_x_coordinate  (stations) float64 40B 106.5 106.7 106.9 107.0 107.2
    station_y_coordinate  (stations) float64 40B -6.021 -6.021 ... -5.962 -5.962
    return_periods        float64 8B 100.0
Dimensions without coordinates: stations
Data variables:
    RPS                   (stations) float64 40B 1.091 1.103 1.105 1.082 1.003
Attributes:
    CDI:            Climate Data Interface version 1.9.5 (http://mpimet.mpg.d...
    Conventions:    CF-1.5 Deltares-0.1
    history:        Thu Sep 19 11:22:14 2019: cdo -L -yearmax -selyear,1979 /...
    source:         Deltares, D-Flow FM Version 1.1.270.54102, Jan 16 2018, 1...
    institution:    Deltares
    references:     http://www.deltares.nl
    date_created:   2019-03-06T17:18:41+0100
    date_modified:  2019-03-06T17:18:41+01

In [ ]:
import pandas as pd

# Ensure rp_100_data is defined, as it was defined in a previous cell but might not be in the current kernel state.
rp_100_data = jakarta_ds.sel(return_periods=100, method='nearest')

# Ensure station_data_rp100 is defined
station_data_rp100 = rp_100_data[['station_name', 'station_x_coordinate', 'station_y_coordinate', 'RPS']]

# Convert the xarray Dataset to a pandas DataFrame
# Ensure the station_name is decoded to a string if it's still in bytes
stations_df = station_data_rp100.to_dataframe()
stations_df['station_name'] = stations_df['station_name'].astype(str)

# Reset index to make 'stations' a column if needed, or work with it as index
stations_df = stations_df.reset_index()

# Select and rename columns for clarity if desired
stations_df = stations_df[['station_name', 'station_x_coordinate', 'station_y_coordinate', 'RPS']]
stations_df = stations_df.rename(columns={'station_x_coordinate': 'Longitude', 'station_y_coordinate': 'Latitude'})

# Define the output CSV filename
output_csv_filename = 'jakarta_nearest_stations_rp100.csv'

# Save the DataFrame to a CSV file
stations_df.to_csv(output_csv_filename, index=False)

print(f"Nearest station data saved to {output_csv_filename}")
print("Contents of the CSV file:")
print(stations_df.head())

Nearest station data saved to jakarta_nearest_stations_rp100.csv
Contents of the CSV file:
          station_name  Longitude  Latitude       RPS
0  id_coast_glob_12946   106.4795 -6.020508  1.090531
1  id_coast_glob_12947   106.7139 -6.020508  1.102772
2  id_coast_glob_12948   106.8896 -6.079101  1.105401
3  id_coast_glob_12949   107.0068 -5.961914  1.081622
4  id_coast_glob_12950   107.2119 -5.961914  1.003216


You can now download the `jakarta_nearest_stations_rp100.csv` file and import it into ArcGIS. When importing, you will typically specify 'Longitude' as the X-coordinate and 'Latitude' as the Y-coordinate.